# Nano Protein AIs — Lecture Figures

Comprehensive visualizations for three minimal protein AI models:
ProteinMPNN, AlphaFold2, and RFDiffusion.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

# Project root
ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT))

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

# Consistent style
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

# Model colors
COLORS = {
    "proteinmpnn": "#DD8452",
    "alphafold2": "#55A868",
    "rfdiffusion": "#C44E52",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Log parser utility
def parse_log(log_path, fields):
    """Parse training log. Returns dict of field_name -> list of floats."""
    data = {f: [] for f in fields}
    pattern = {f: re.compile(rf"{f}=([0-9eE.+-]+)") for f in fields}
    with open(log_path) as fh:
        for line in fh:
            if "step=" not in line:
                continue
            for f in fields:
                m = pattern[f].search(line)
                if m:
                    data[f].append(float(m.group(1)))
    return {f: np.array(v) for f, v in data.items()}

## Section 1: Training Curves

In [ ]:
# Parse all training logs
mpnn_log = parse_log(ROOT / "outputs/proteinmpnn/train.log", ["step", "loss", "recovery"])
af2_log = parse_log(ROOT / "outputs/alphafold2/train.log", ["step", "loss", "fape"])
rfd_log = parse_log(ROOT / "outputs/rfdiffusion/train.log", ["step", "loss", "trans", "rot"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ProteinMPNN: loss + recovery (dual y-axis)
ax = axes[0]
ax.plot(mpnn_log["step"], mpnn_log["loss"], color=COLORS["proteinmpnn"], label="Loss")
ax.set_xlabel("Step")
ax.set_ylabel("Loss", color=COLORS["proteinmpnn"])
ax.tick_params(axis="y", labelcolor=COLORS["proteinmpnn"])
ax2 = ax.twinx()
ax2.plot(mpnn_log["step"], mpnn_log["recovery"] * 100, color="#E8A97E", linestyle="--", label="Recovery")
ax2.set_ylabel("Recovery (%)", color="#E8A97E")
ax2.tick_params(axis="y", labelcolor="#E8A97E")
ax.set_title("ProteinMPNN — Inverse Folding")

# AlphaFold2: FAPE loss
ax = axes[1]
ax.plot(af2_log["step"], af2_log["fape"], color=COLORS["alphafold2"])
ax.set_xlabel("Step")
ax.set_ylabel("FAPE Loss")
ax.set_title("AlphaFold2 — Structure Prediction")

# RFDiffusion: trans + rot loss (log y-scale)
ax = axes[2]
ax.semilogy(rfd_log["step"], rfd_log["trans"], color=COLORS["rfdiffusion"], alpha=0.7, label="Translation")
ax.semilogy(rfd_log["step"], rfd_log["rot"], color="#D4787A", alpha=0.7, label="Rotation")
ax.set_xlabel("Step")
ax.set_ylabel("Loss (log scale)")
ax.set_title("RFDiffusion — Backbone Generation")
ax.legend(fontsize=10)

fig.suptitle("Training Curves", fontsize=16, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES / "fig1_training_curves.png")
plt.show()
print("Saved fig1_training_curves.png")

## Section 2: ProteinMPNN — Inverse Folding

In [ ]:
from proteinmpnn.model import ProteinMPNN
from proteinmpnn.train import parse_pdb, AA_VOCAB

mpnn_model = ProteinMPNN().to(device)
ckpt = torch.load(ROOT / "outputs/proteinmpnn/final_model.pt", map_location=device, weights_only=True)
mpnn_model.load_state_dict(ckpt["model_state_dict"])
mpnn_model.eval()
print(f"ProteinMPNN loaded: {mpnn_model.count_parameters():,} parameters")

crn_pdb = parse_pdb(ROOT / "data/pdb/1CRN.pdb")
crn_N = crn_pdb["coords_N"].to(device)
crn_CA = crn_pdb["coords_CA"].to(device)
crn_C = crn_pdb["coords_C"].to(device)
crn_O = crn_pdb["coords_O"].to(device)
crn_pdb_mask = crn_pdb["mask"].to(device)
crn_native_seq = crn_pdb["sequence"]
L_crn = crn_CA.shape[0]
print(f"1CRN: {L_crn} residues")

In [ ]:
# Fig 2a: 3D backbone of 1CRN
ca = crn_CA.cpu().numpy()

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
colors = plt.cm.viridis(np.linspace(0, 1, L_crn))
for i in range(L_crn - 1):
    ax.plot(ca[i:i+2, 0], ca[i:i+2, 1], ca[i:i+2, 2], color=colors[i], linewidth=2)
ax.scatter(*ca[0], color="blue", s=80, zorder=5)
ax.scatter(*ca[-1], color="red", s=80, zorder=5)
ax.text(*ca[0], "  N", fontsize=11, color="blue")
ax.text(*ca[-1], "  C", fontsize=11, color="red")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("1CRN Backbone (CA trace)")
fig.tight_layout()
fig.savefig(FIGURES / "fig2a_proteinmpnn_backbone_3d.png")
plt.show()
print("Saved fig2a_proteinmpnn_backbone_3d.png")

In [ ]:
# Fig 2b: Sequence alignment heatmap
torch.manual_seed(42)
designs = mpnn_model.design(crn_N, crn_CA, crn_C, crn_O, crn_pdb_mask,
                            temperature=0.1, num_samples=5)  # [5, L]

native = crn_native_seq.cpu().numpy()
all_seqs = np.vstack([native[None, :], designs.cpu().numpy()])  # [6, L]
match_matrix = (all_seqs == native[None, :]).astype(float)  # [6, L]

recoveries = [100.0]  # native = 100%
for i in range(5):
    rec = (designs[i].cpu().numpy() == native).mean() * 100
    recoveries.append(rec)

fig, ax = plt.subplots(figsize=(14, 3))
cmap = matplotlib.colors.ListedColormap(["#E57373", "#81C784"])
ax.imshow(match_matrix, cmap=cmap, aspect="auto", interpolation="nearest")
row_labels = [f"Native (100.0%)"]
for i in range(5):
    row_labels.append(f"Design {i+1} ({recoveries[i+1]:.1f}%)")
ax.set_yticks(range(6))
ax.set_yticklabels(row_labels, fontsize=9)
ax.set_xlabel("Residue position")
ax.set_title("ProteinMPNN Sequence Design — 1CRN (green=match, red=mismatch)")
fig.tight_layout()
fig.savefig(FIGURES / "fig2b_proteinmpnn_alignment.png")
plt.show()
print("Saved fig2b_proteinmpnn_alignment.png")

In [ ]:
# Fig 2c: Recovery vs temperature
temperatures = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
n_samples = 10
mean_recs = []
std_recs = []

for temp in temperatures:
    torch.manual_seed(0)
    seqs = mpnn_model.design(crn_N, crn_CA, crn_C, crn_O, crn_pdb_mask,
                             temperature=temp, num_samples=n_samples)
    recs = (seqs.cpu().numpy() == native[None, :]).mean(axis=1) * 100
    mean_recs.append(recs.mean())
    std_recs.append(recs.std())

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(temperatures, mean_recs, yerr=std_recs, fmt="o-", color=COLORS["proteinmpnn"],
            capsize=4, linewidth=2, markersize=7)
ax.axhline(y=100/20, color="gray", linestyle="--", alpha=0.6, label="Random (5%)")
ax.set_xscale("log")
ax.set_xlabel("Temperature")
ax.set_ylabel("Sequence Recovery (%)")
ax.set_title("ProteinMPNN Recovery vs Temperature — 1CRN")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "fig2c_proteinmpnn_recovery_vs_temp.png")
plt.show()
print("Saved fig2c_proteinmpnn_recovery_vs_temp.png")

## Section 3: AlphaFold2 — Structure Prediction

In [ ]:
from alphafold2.model import AlphaFold2

af2_model = AlphaFold2().to(device)
ckpt = torch.load(ROOT / "outputs/alphafold2/final_model.pt", map_location=device, weights_only=True)
af2_model.load_state_dict(ckpt["model_state_dict"])
af2_model.eval()
print(f"AlphaFold2 loaded: {af2_model.count_parameters():,} parameters")

# Predict 1CRN structure
crn_seq_tensor = crn_pdb["sequence"].to(device)
with torch.no_grad():
    af2_pred = af2_model.predict(crn_seq_tensor)
pred_coords = af2_pred["coords"].cpu().numpy()  # [L, 3]
plddt = af2_pred["plddt"].cpu().numpy()          # [L]
true_ca = crn_CA.cpu().numpy()                    # [L, 3]
print(f"Predicted coords: {pred_coords.shape}, pLDDT range: [{plddt.min():.3f}, {plddt.max():.3f}]")

In [ ]:
# Fig 3a: Predicted vs true overlay
# Center both structures
pred_centered = pred_coords - pred_coords.mean(axis=0)
true_centered = true_ca - true_ca.mean(axis=0)
rmsd = np.sqrt(((pred_centered - true_centered) ** 2).sum(axis=1).mean())

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(*true_centered.T, color="#555555", linewidth=2, label="True", alpha=0.8)
ax.plot(*pred_centered.T, color=COLORS["alphafold2"], linewidth=2, label="Predicted", alpha=0.8)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title(f"AlphaFold2 — 1CRN Predicted vs True (CA-RMSD: {rmsd:.1f} A)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "fig3a_alphafold2_overlay.png")
plt.show()
print("Saved fig3a_alphafold2_overlay.png")

In [ ]:
# Fig 3b: Per-residue pLDDT
fig, ax = plt.subplots(figsize=(12, 4))
colors_plddt = []
for v in plddt:
    if v > 0.9:
        colors_plddt.append("#1565C0")    # blue
    elif v > 0.7:
        colors_plddt.append("#4DD0E1")    # cyan
    elif v > 0.5:
        colors_plddt.append("#FFD54F")    # yellow
    else:
        colors_plddt.append("#FF8A65")    # orange

ax.bar(range(L_crn), plddt, color=colors_plddt, width=0.9)
ax.axhline(y=0.9, color="#1565C0", linestyle="--", alpha=0.3)
ax.axhline(y=0.7, color="#4DD0E1", linestyle="--", alpha=0.3)
ax.axhline(y=0.5, color="#FFD54F", linestyle="--", alpha=0.3)
ax.set_xlabel("Residue")
ax.set_ylabel("pLDDT")
ax.set_ylim(0, 1)
ax.set_title("AlphaFold2 Per-Residue Confidence (pLDDT) — 1CRN")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#1565C0", label="Very high (>0.9)"),
    Patch(facecolor="#4DD0E1", label="High (0.7-0.9)"),
    Patch(facecolor="#FFD54F", label="Low (0.5-0.7)"),
    Patch(facecolor="#FF8A65", label="Very low (<0.5)"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "fig3b_alphafold2_plddt.png")
plt.show()
print("Saved fig3b_alphafold2_plddt.png")

In [ ]:
# Fig 3c: Distance matrices
from scipy.spatial.distance import cdist

true_dist = cdist(true_centered, true_centered)
pred_dist = cdist(pred_centered, pred_centered)
vmin, vmax = 0, max(true_dist.max(), pred_dist.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
im1 = ax1.imshow(true_dist, cmap="viridis", vmin=vmin, vmax=vmax)
ax1.set_title("True CA Distance Matrix")
ax1.set_xlabel("Residue")
ax1.set_ylabel("Residue")

im2 = ax2.imshow(pred_dist, cmap="viridis", vmin=vmin, vmax=vmax)
ax2.set_title("Predicted CA Distance Matrix")
ax2.set_xlabel("Residue")
ax2.set_ylabel("Residue")

fig.colorbar(im2, ax=[ax1, ax2], shrink=0.8, label="Distance (A)")
fig.suptitle("AlphaFold2 — 1CRN Distance Matrices", fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES / "fig3c_alphafold2_distance_matrix.png")
plt.show()
print("Saved fig3c_alphafold2_distance_matrix.png")

## Section 4: RFDiffusion — Backbone Generation

In [ ]:
from rfdiffusion.model import RFDiffusion, SE3Diffusion, sample

rfd_model = RFDiffusion().to(device)
ckpt = torch.load(ROOT / "outputs/rfdiffusion/final_model.pt", map_location=device, weights_only=True)
rfd_model.load_state_dict(ckpt["model_state_dict"])
rfd_model.eval()
print(f"RFDiffusion loaded: {rfd_model.count_parameters():,} parameters")

# Sample a 50-residue backbone
torch.manual_seed(42)
diffusion = SE3Diffusion()
trajectory = sample(rfd_model.network, diffusion, num_residues=50, num_steps=100, device=device)
print(f"Generated trajectory: {len(trajectory)} steps, {trajectory[-1].trans.shape[0]} residues")

In [ ]:
# Fig 4a: Diffusion trajectory
step_indices = [0, 25, 50, 75, 100]

fig, axes = plt.subplots(1, 5, figsize=(20, 4), subplot_kw={"projection": "3d"})
for ax, si in zip(axes, step_indices):
    ca = trajectory[si].trans.cpu().numpy()  # [L, 3]
    ca_c = ca - ca.mean(axis=0)
    ax.plot(*ca_c.T, linewidth=1.5, color=COLORS["rfdiffusion"])
    ax.scatter(*ca_c[0], color="blue", s=30, zorder=5)
    ax.scatter(*ca_c[-1], color="red", s=30, zorder=5)
    ax.set_title(f"Step {si}", fontsize=11)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])

fig.suptitle("RFDiffusion — Reverse Diffusion Trajectory (noise to structure)", fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES / "fig4a_rfdiffusion_trajectory.png")
plt.show()
print("Saved fig4a_rfdiffusion_trajectory.png")

In [ ]:
# Fig 4b: Bond length histogram
gen_ca = trajectory[-1].trans.cpu().numpy()  # [50, 3]
gen_bonds = np.linalg.norm(np.diff(gen_ca, axis=0), axis=1)

# Real bond lengths from all 9 PDBs
real_bonds = []
for pdb_file in sorted((ROOT / "data/pdb").glob("*.pdb")):
    pdb_data = parse_pdb(pdb_file)
    if pdb_data is not None:
        ca_coords = pdb_data["coords_CA"].numpy()
        bonds = np.linalg.norm(np.diff(ca_coords, axis=0), axis=1)
        real_bonds.append(bonds)
real_bonds = np.concatenate(real_bonds)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(real_bonds, bins=40, alpha=0.6, color="#555555", label="Real (9 PDBs)", density=True)
ax.hist(gen_bonds, bins=20, alpha=0.6, color=COLORS["rfdiffusion"], label="Generated", density=True)
ax.axvline(x=3.8, color="black", linestyle="--", linewidth=1.5, label="Ideal CA-CA (3.8 A)")
ax.set_xlabel("CA-CA Bond Length (A)")
ax.set_ylabel("Density")
ax.set_title("RFDiffusion — CA-CA Bond Length Distribution")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "fig4b_rfdiffusion_bond_lengths.png")
plt.show()
print("Saved fig4b_rfdiffusion_bond_lengths.png")

In [ ]:
# Fig 4c: Generated backbone 3D
gen_ca_c = gen_ca - gen_ca.mean(axis=0)
L_gen = len(gen_ca_c)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
colors = plt.cm.plasma(np.linspace(0, 1, L_gen))
for i in range(L_gen - 1):
    ax.plot(gen_ca_c[i:i+2, 0], gen_ca_c[i:i+2, 1], gen_ca_c[i:i+2, 2],
            color=colors[i], linewidth=2)
ax.scatter(*gen_ca_c[0], color="blue", s=80, zorder=5)
ax.scatter(*gen_ca_c[-1], color="red", s=80, zorder=5)
ax.text(*gen_ca_c[0], "  N", fontsize=11, color="blue")
ax.text(*gen_ca_c[-1], "  C", fontsize=11, color="red")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("RFDiffusion — Generated Backbone (50 residues)")
fig.tight_layout()
fig.savefig(FIGURES / "fig4c_rfdiffusion_generated_backbone.png")
plt.show()
print("Saved fig4c_rfdiffusion_generated_backbone.png")

## Section 5: Model Comparison Summary

In [ ]:
# Fig 5: Model comparison
model_names = ["ProteinMPNN", "AlphaFold2", "RFDiffusion"]
param_counts = [
    mpnn_model.count_parameters(),
    af2_model.count_parameters(),
    rfd_model.count_parameters(),
]
color_list = [COLORS["proteinmpnn"], COLORS["alphafold2"], COLORS["rfdiffusion"]]

# Summary data
tasks = ["Inverse Folding", "Structure Pred.", "Backbone Gen."]
final_metrics = [
    f"loss={mpnn_log['loss'][-1]:.3f}, rec={mpnn_log['recovery'][-1]*100:.1f}%",
    f"FAPE={af2_log['fape'][-1]:.2f}",
    f"trans={rfd_log['trans'][-1]:.1f}, rot={rfd_log['rot'][-1]:.2f}",
]
epochs = ["100", "100", "150"]

fig, (ax_bar, ax_table) = plt.subplots(1, 2, figsize=(14, 4),
                                        gridspec_kw={"width_ratios": [1, 1.5]})

# Left: parameter count bar chart
bars = ax_bar.barh(model_names, [p / 1e6 for p in param_counts], color=color_list)
ax_bar.set_xlabel("Parameters (millions)")
ax_bar.set_title("Model Size")
for bar, p in zip(bars, param_counts):
    ax_bar.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
               f"{p/1e6:.1f}M", va="center", fontsize=10)
ax_bar.set_xlim(0, max(p / 1e6 for p in param_counts) * 1.25)

# Right: summary table
ax_table.axis("off")
table_data = []
for i in range(3):
    table_data.append([model_names[i], tasks[i], f"{param_counts[i]/1e6:.1f}M",
                       final_metrics[i], epochs[i]])
table = ax_table.table(
    cellText=table_data,
    colLabels=["Model", "Task", "Params", "Final Metric", "Epochs"],
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.8)
# Color header row
for j in range(5):
    table[0, j].set_facecolor("#E8E8E8")
    table[0, j].set_text_props(weight="bold")
# Color model name cells
for i in range(3):
    table[i + 1, 0].set_facecolor(color_list[i] + "33")  # alpha via hex

fig.suptitle("Nano Protein AIs — Model Comparison", fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES / "fig5_model_comparison.png")
plt.show()
print("Saved fig5_model_comparison.png")

In [ ]:
# Verify all figures were saved
expected = [
    "fig1_training_curves.png",
    "fig2a_proteinmpnn_backbone_3d.png",
    "fig2b_proteinmpnn_alignment.png",
    "fig2c_proteinmpnn_recovery_vs_temp.png",
    "fig3a_alphafold2_overlay.png",
    "fig3b_alphafold2_plddt.png",
    "fig3c_alphafold2_distance_matrix.png",
    "fig4a_rfdiffusion_trajectory.png",
    "fig4b_rfdiffusion_bond_lengths.png",
    "fig4c_rfdiffusion_generated_backbone.png",
    "fig5_model_comparison.png",
]
for fname in expected:
    path = FIGURES / fname
    status = "OK" if path.exists() else "MISSING"
    size = f"{path.stat().st_size / 1024:.0f}KB" if path.exists() else "-"
    print(f"  [{status}] {fname} ({size})")

found = sum(1 for f in expected if (FIGURES / f).exists())
print(f"\n{found}/{len(expected)} figures saved to {FIGURES.resolve()}")